# SQLite database: electricity prices and municipality data

**ZHAW Data Analytics HS2026 · Group project “Electricity prices of the municipalities” · Part Metehan (Requirement 3)**

This notebook builds an SQLite database `strompreise.db` from the CSV files and queries it with SQL from Python.

**Input files** (in the folder `daten/`):

| File | Content | Source |
| --- | --- | --- |
| `Daten_Strompreise_2021.csv`, `_2024.csv`, `_2027.csv` | All tariffs per municipality, grid operator, product and category | ElCom via LINDAS (SPARQL) |
| `regionalportraets_2021.csv` | Indicators per municipality in long format (one value per row) | FSO Regional Portraits 2021 |
| `strompreise_gemeinden.csv` | Lenny's joined table, only for the check at the end | own join |

**Data model.** Instead of putting everything into one large table, each thing is stored only once:

```
gemeinde (gemeinde_nr PK, name)
   │1
   │n
tarif (jahr, gemeinde_nr FK, netzbetreiber_id FK, produkt, kategorie,  ← PK from these 5 columns
       total, energie, netznutzung, abgaben, netzzuschlag, messtarif, messtarif_chf_jahr)
   │n
   │1
netzbetreiber (netzbetreiber_id PK, name)

gemeinde 1 ── n  kennzahl (gemeinde_nr FK, indikator FK, wert, status)  n ── 1  indikator (code PK, bezeichnung, einheit, periode)
```

Two **views** build on this (stored queries, not copies of the data):
- `v_preis_h4`: one price per municipality and year (standard product, category H4, mean over the grid operators)
- `v_analyse`: one row per municipality with prices 2021/2024/2027 and municipal characteristics, the table for EDA and the model

*Identifiers stay German: table, column, view and variable names are shared with the CSV files, the SQLite and PostgreSQL databases and the other notebooks, so only text and comments are translated.*

In [1]:
import sqlite3
from pathlib import Path

import pandas as pd

def finde_wurzel(start: Path) -> Path:
    """Climbs upwards until daten/ and scripts/ sit side by side."""
    for ordner in [start, *start.parents]:
        if (ordner / "daten").is_dir() and (ordner / "scripts").is_dir():
            return ordner
    raise RuntimeError(f"Repository root not found from {start}")


WURZEL = finde_wurzel(Path.cwd().resolve())

DATEN = WURZEL / "daten"
DB = WURZEL / "sql" / "strompreise.db"
DB.parent.mkdir(parents=True, exist_ok=True)
JAHRE = [2021, 2024, 2027]

# Rebuild on every run so that the notebook is reproducible
if DB.exists():
    DB.unlink()

con = sqlite3.connect(DB)
con.execute("PRAGMA foreign_keys = ON")   # SQLite only checks foreign keys with this switch
print("SQLite version:", sqlite3.sqlite_version)

SQLite version: 3.53.1


## 1 · Create the tables

`CREATE TABLE` defines the structure before any data goes in. Three things matter:

- **Primary keys (PK)** prevent duplicate rows. A tariff is uniquely identified by year, municipality, grid operator, product and category.
- **Foreign keys (FK)** make sure that every municipality and grid operator number in `tarif` also exists in its own table.
- **CHECK** catches impossible values, for example a wrong product.

In [2]:
SCHEMA = '''
CREATE TABLE gemeinde (
    gemeinde_nr      INTEGER PRIMARY KEY,      -- FSO municipality number
    name             TEXT                      -- from the FSO, otherwise from ElCom
);

CREATE TABLE netzbetreiber (
    netzbetreiber_id INTEGER PRIMARY KEY,      -- ElCom number of the grid operator
    name             TEXT NOT NULL
);

CREATE TABLE tarif (
    jahr               INTEGER NOT NULL,
    gemeinde_nr        INTEGER NOT NULL REFERENCES gemeinde(gemeinde_nr),
    netzbetreiber_id   INTEGER NOT NULL REFERENCES netzbetreiber(netzbetreiber_id),
    produkt            TEXT    NOT NULL CHECK (produkt IN ('standard', 'cheapest')),
    kategorie          TEXT    NOT NULL,       -- H1-H8 households, C1-C7 businesses
    total              REAL    NOT NULL,       -- all prices in Rp./kWh, excl. VAT
    energie            REAL,
    netznutzung        REAL,
    abgaben            REAL,
    netzzuschlag       REAL,
    messtarif          REAL    NOT NULL DEFAULT 0,  -- separate only from 2026, before 0
    messtarif_chf_jahr REAL,                   -- CHF per year, information only
    PRIMARY KEY (jahr, gemeinde_nr, netzbetreiber_id, produkt, kategorie)
);

CREATE TABLE indikator (
    code         TEXT PRIMARY KEY,             -- e.g. Ind_01_01
    bezeichnung  TEXT NOT NULL,
    einheit      TEXT,
    periode      TEXT,                         -- reference year of the indicator at the FSO
    label_quelle TEXT                          -- 'geprueft' (checked) or 'abgeleitet' (derived), see section 3
);

CREATE TABLE kennzahl (
    gemeinde_nr INTEGER NOT NULL REFERENCES gemeinde(gemeinde_nr),
    indikator   TEXT    NOT NULL REFERENCES indikator(code),
    wert        REAL,                          -- empty if the FSO reports no value
    status      TEXT,                          -- FSO status: A = value available, M/Q = no value
    PRIMARY KEY (gemeinde_nr, indikator)
);

-- Index for the most frequent query: one product and one category across all municipalities
CREATE INDEX idx_tarif_produkt_kat ON tarif (produkt, kategorie, jahr);
'''
con.executescript(SCHEMA)
pd.read_sql("SELECT name, type FROM sqlite_master WHERE type IN ('table', 'index') ORDER BY type DESC, name", con)

,name,type
0,gemeinde,table
1,indikator,table
2,kennzahl,table
3,netzbetreiber,table
4,tarif,table
5,idx_tarif_produkt_kat,index
6,sqlite_autoindex_indikator_1,index
7,sqlite_autoindex_kennzahl_1,index
8,sqlite_autoindex_tarif_1,index


## 2 · Load the electricity prices

The three yearly files are read with pandas and concatenated. Three tables are created from them:

- `netzbetreiber`: each number once, with the name from the most recent year
- `gemeinde`: each number once. The name comes from the FSO first, otherwise from the most recent ElCom year. This way the 120 tariff rows from 2021 for which ElCom supplies no name (merged municipalities) also get a name. No number is lost.
- `tarif`: all price rows, without the names (they are in the other tables)

Rows are inserted with `executemany` and `INSERT`, i.e. with real SQL. `pandas.to_sql` would create the tables itself and overwrite the schema with its keys.

In [3]:
preise = pd.concat(
    [pd.read_csv(DATEN / "strompreise" / f"Daten_Strompreise_{j}.csv") for j in JAHRE],
    ignore_index=True,
)
preise["messtarif"] = preise["messtarif"].fillna(0.0)
print(f"{len(preise):,} price rows read")
preise.groupby("jahr").size().rename("zeilen").to_frame()

143,837 price rows read


,zeilen
jahr,
2021,54210
2024,48187
2027,41440


In [4]:
# Read the FSO data already now, because the municipality names come from there
bfs = pd.read_csv(DATEN / "bfs" / "regionalportraets_2021.csv", sep=";", encoding="utf-8-sig", dtype=str)
bfs = bfs[bfs["CODE_REGION"] != "CH"].copy()          # drop the row for Switzerland as a whole
bfs["gemeinde_nr"] = bfs["CODE_REGION"].astype(int)
bfs["wert"] = pd.to_numeric(bfs["VALUE"], errors="coerce")

# Grid operators: name from the most recent year
nb = (preise.sort_values("jahr")
            .drop_duplicates("netzbetreiber_id", keep="last")[["netzbetreiber_id", "netzbetreiber"]])

# Municipalities: name from the FSO first, otherwise from the most recent ElCom year
namen_bfs = bfs.drop_duplicates("gemeinde_nr").set_index("gemeinde_nr")["REGION"]
namen_elcom = (preise.dropna(subset=["gemeinde"]).sort_values("jahr")
                     .drop_duplicates("gemeinde_nr", keep="last").set_index("gemeinde_nr")["gemeinde"])
alle_nr = sorted(set(preise["gemeinde_nr"]) | set(bfs["gemeinde_nr"]))
gem = pd.DataFrame({"gemeinde_nr": alle_nr})
gem["name"] = gem["gemeinde_nr"].map(namen_bfs).fillna(gem["gemeinde_nr"].map(namen_elcom))

def zeilen(df):
    # convert pandas values to Python types, missing values as None (= NULL in SQL)
    return [tuple(None if pd.isna(v) else (v.item() if hasattr(v, "item") else v) for v in row)
            for row in df.itertuples(index=False)]

con.executemany("INSERT INTO netzbetreiber VALUES (?, ?)", zeilen(nb))
con.executemany("INSERT INTO gemeinde VALUES (?, ?)", zeilen(gem))

tarif_spalten = ["jahr", "gemeinde_nr", "netzbetreiber_id", "produkt", "kategorie", "total", "energie",
                 "netznutzung", "abgaben", "netzzuschlag", "messtarif", "messtarif_chf_jahr"]
con.executemany(f"INSERT INTO tarif VALUES ({', '.join('?' * len(tarif_spalten))})",
                zeilen(preise[tarif_spalten]))
con.commit()

print(f"netzbetreiber: {len(nb):,} · gemeinde: {len(gem):,} "
      f"(of which without a name: {gem['name'].isna().sum()}) · tarif: {len(preise):,}")

netzbetreiber: 609 · gemeinde: 2,225 (of which without a name: 0) · tarif: 143,837


## 3 · Load the FSO municipality data

The Regional Portraits come in **long format**: one row per municipality and indicator. They stay that way in the database. This is flexible, because new indicators need no new column. A view later creates the wide form for the analysis.

The CSV contains only codes (`Ind_01_01`). The labels are in the table `indikator`.

**Caution with the labels:** Five codes are `geprueft` (checked), because their values match Lenny's columns exactly (population, density, area, employees, share 65+). The rest are `abgeleitet` (derived): they were inferred from the order of the FSO topics and from plausibility checks, for example Zurich with 0.15 % vacant dwellings and 32 % foreign nationals. Before you use a derived indicator in the model, compare its label with the header row of the FSO Excel or ODS file.

In [5]:
# (code, bezeichnung = label, einheit = unit, label_quelle = label source)
# The labels are stored as data in the database and therefore stay German.
INDIKATOREN = [
    ("Ind_01_01", "Einwohner",                                   "Anzahl",           "geprueft"),
    ("Ind_01_02", "Veränderung der Einwohnerzahl",               "%",                "abgeleitet"),
    ("Ind_01_03", "Bevölkerungsdichte",                          "Einwohner pro km²", "geprueft"),
    ("Ind_01_04", "Anteil 0-19 Jahre",                           "%",                "abgeleitet"),
    ("Ind_01_05", "Anteil 20-64 Jahre",                          "%",                "abgeleitet"),
    ("Ind_01_06", "Anteil 65 Jahre und älter",                   "%",                "geprueft"),
    ("Ind_01_08", "Ausländeranteil",                             "%",                "abgeleitet"),
    ("Ind_01_09", "Rohe Heiratsziffer",                          "pro 1000 Einwohner", "abgeleitet"),
    ("Ind_01_10", "Rohe Scheidungsziffer",                       "pro 1000 Einwohner", "abgeleitet"),
    ("Ind_01_11", "Rohe Geburtenziffer",                         "pro 1000 Einwohner", "abgeleitet"),
    ("Ind_01_12", "Rohe Sterbeziffer",                           "pro 1000 Einwohner", "abgeleitet"),
    ("Ind_01_13", "Privathaushalte",                             "Anzahl",           "abgeleitet"),
    ("Ind_01_14", "Durchschnittliche Haushaltsgrösse",           "Personen",         "abgeleitet"),
    ("Ind_04_01", "Gesamtfläche",                                "km²",              "geprueft"),
    ("Ind_04_02", "Siedlungsfläche",                             "%",                "abgeleitet"),
    ("Ind_04_03", "Veränderung Siedlungsfläche",                 "ha",               "abgeleitet"),
    ("Ind_04_04", "Landwirtschaftsfläche",                       "%",                "abgeleitet"),
    ("Ind_04_05", "Veränderung Landwirtschaftsfläche",           "ha",               "abgeleitet"),
    ("Ind_04_06", "Wald und Gehölze",                            "%",                "abgeleitet"),
    ("Ind_04_07", "Unproduktive Fläche",                         "%",                "abgeleitet"),
    ("Ind_06_03", "Beschäftigte total",                          "Anzahl",           "geprueft"),
    ("Ind_06_04", "Beschäftigte 1. Sektor",                      "Anzahl",           "abgeleitet"),
    ("Ind_06_05", "Beschäftigte 2. Sektor",                      "Anzahl",           "abgeleitet"),
    ("Ind_06_06", "Beschäftigte 3. Sektor",                      "Anzahl",           "abgeleitet"),
    ("Ind_06_07", "Arbeitsstätten total",                        "Anzahl",           "abgeleitet"),
    ("Ind_06_08", "Arbeitsstätten 1. Sektor",                    "Anzahl",           "abgeleitet"),
    ("Ind_06_09", "Arbeitsstätten 2. Sektor",                    "Anzahl",           "abgeleitet"),
    ("Ind_06_10", "Arbeitsstätten 3. Sektor",                    "Anzahl",           "abgeleitet"),
    ("Ind_08_01", "Leerwohnungsziffer",                          "%",                "abgeleitet"),
    ("Ind_08_04", "Neu gebaute Wohnungen",                       "pro 1000 Einwohner", "abgeleitet"),
    ("Ind_11_01", "Sozialhilfequote",                            "%",                "abgeleitet"),
    ("Ind_14_01", "Wähleranteil FDP (Nationalrat 2019)",         "%",                "abgeleitet"),
    ("Ind_14_02", "Wähleranteil CVP (Nationalrat 2019)",         "%",                "abgeleitet"),
    ("Ind_14_03", "Wähleranteil SP (Nationalrat 2019)",          "%",                "abgeleitet"),
    ("Ind_14_04", "Wähleranteil SVP (Nationalrat 2019)",         "%",                "abgeleitet"),
    ("Ind_14_05", "Wähleranteil EVP/CSP (Nationalrat 2019)",     "%",                "abgeleitet"),
    ("Ind_14_06", "Wähleranteil GLP (Nationalrat 2019)",         "%",                "abgeleitet"),
    ("Ind_14_07", "Wähleranteil BDP (Nationalrat 2019)",         "%",                "abgeleitet"),
    ("Ind_14_08", "Wähleranteil PdA/Sol. (Nationalrat 2019)",    "%",                "abgeleitet"),
    ("Ind_14_09", "Wähleranteil Grüne (Nationalrat 2019)",       "%",                "abgeleitet"),
    ("Ind_14_10", "Wähleranteil kleine Rechtsparteien (Nationalrat 2019)", "%",      "abgeleitet"),
]
periode = bfs.groupby("INDICATORS")["PERIOD_REF"].first()
ind = pd.DataFrame(INDIKATOREN, columns=["code", "bezeichnung", "einheit", "label_quelle"])
ind["periode"] = ind["code"].map(periode)

fehlend = set(bfs["INDICATORS"]) - set(ind["code"])
assert not fehlend, f"Codes without a label: {fehlend}"

con.executemany("INSERT INTO indikator (code, bezeichnung, einheit, label_quelle, periode) VALUES (?, ?, ?, ?, ?)",
                zeilen(ind[["code", "bezeichnung", "einheit", "label_quelle", "periode"]]))
con.executemany("INSERT INTO kennzahl VALUES (?, ?, ?, ?)",
                zeilen(bfs[["gemeinde_nr", "INDICATORS", "wert", "STATUS"]]))
con.commit()
print(f"indikator: {len(ind)} · kennzahl: {len(bfs):,} rows for {bfs['gemeinde_nr'].nunique():,} municipalities")

indikator: 41 · kennzahl: 89,052 rows for 2,172 municipalities


## 4 · Checks

Before anyone calculates with the database, we check whether anything was lost or broken during loading.

In [6]:
pd.read_sql('''
    SELECT 'gemeinde'      AS tabelle, COUNT(*) AS zeilen FROM gemeinde
    UNION ALL SELECT 'netzbetreiber', COUNT(*) FROM netzbetreiber
    UNION ALL SELECT 'tarif',         COUNT(*) FROM tarif
    UNION ALL SELECT 'indikator',     COUNT(*) FROM indikator
    UNION ALL SELECT 'kennzahl',      COUNT(*) FROM kennzahl
''', con)

,tabelle,zeilen
0,gemeinde,2225
1,netzbetreiber,609
2,tarif,143837
3,indikator,41
4,kennzahl,89052


In [7]:
# a) Foreign keys: returns no rows if all references are correct
fk_fehler = con.execute("PRAGMA foreign_key_check").fetchall()
print("Violated foreign keys:", len(fk_fehler))

# b) Sum check directly in SQL: the components must add up to the total
pd.read_sql('''
    SELECT jahr,
           COUNT(*) AS zeilen,
           SUM(ABS(total - (energie + netznutzung + abgaben + netzzuschlag + messtarif)) > 0.01) AS summe_falsch
    FROM tarif
    GROUP BY jahr
''', con)

Violated foreign keys: 0


,jahr,zeilen,summe_falsch
0,2021,54210,0
1,2024,48187,0
2,2027,41440,0


## 5 · Views for the analysis

**`v_preis_h4`**: According to the project plan, we calculate with the standard product and category H4 (4500 kWh per year). If a municipality has several grid operators, the **mean** is taken, as in Lenny's join. `anzahl_netzbetreiber` shows where this happens.

From 2026 the metering tariff is listed separately; before, it was contained in grid usage. To make the years comparable, there is the column `netznutzung_inkl_mess`.

**`v_analyse`**: One row per municipality that has a price in all three years and has FSO data. The indicators are pivoted from long to wide format with `MAX(CASE WHEN ...)`.

**Placeholder rows.** In 2021, four grid operators reported H4 rows with a grid usage charge of 0. An H4 household tariff without grid usage cannot exist, and the same operators report real grid usage in 2024 and 2027. Where such a row sits next to a real tariff in the same municipality and year, `v_preis_h4` leaves it out (10 rows), otherwise the mean would halve the price. Sufers, where the zero row is the only one, is kept so that the analysis set stays at 2,087 municipalities. The table `tarif` is not changed; see `dokumentation_integration_lenny.md`, section 6b.

In [8]:
con.executescript('''
DROP VIEW IF EXISTS v_preis_h4;
CREATE VIEW v_preis_h4 AS
SELECT jahr,
       gemeinde_nr,
       COUNT(*)                     AS anzahl_netzbetreiber,
       AVG(total)                   AS total,
       AVG(energie)                 AS energie,
       AVG(netznutzung + messtarif) AS netznutzung_inkl_mess,
       AVG(abgaben)                 AS abgaben,
       AVG(netzzuschlag)            AS netzzuschlag
FROM tarif t
WHERE produkt = 'standard' AND kategorie = 'H4'
  -- Placeholder rows (2021): grid usage 0 next to a real tariff of another operator
  -- in the same municipality and year. They would halve the mean, so they are left
  -- out here. A zero row that is the municipality's only row (Sufers) stays, so that
  -- no municipality drops out. The table tarif itself is unchanged.
  AND NOT (t.netznutzung = 0 AND EXISTS (
        SELECT 1 FROM tarif t2
        WHERE t2.jahr = t.jahr AND t2.gemeinde_nr = t.gemeinde_nr
          AND t2.produkt = 'standard' AND t2.kategorie = 'H4'
          AND t2.netzbetreiber_id <> t.netzbetreiber_id
          AND t2.netznutzung <> 0))
GROUP BY jahr, gemeinde_nr;

DROP VIEW IF EXISTS v_analyse;
CREATE VIEW v_analyse AS
WITH merkmale AS (
    SELECT gemeinde_nr,
           MAX(CASE WHEN indikator = 'Ind_01_01' THEN wert END) AS einwohner,
           MAX(CASE WHEN indikator = 'Ind_01_03' THEN wert END) AS dichte,
           MAX(CASE WHEN indikator = 'Ind_04_01' THEN wert END) AS flaeche_total,
           MAX(CASE WHEN indikator = 'Ind_06_03' THEN wert END) AS beschaeftigte,
           MAX(CASE WHEN indikator = 'Ind_01_06' THEN wert END) AS anteil_65plus
    FROM kennzahl
    GROUP BY gemeinde_nr
)
SELECT g.gemeinde_nr,
       g.name                 AS gemeinde,
       p21.anzahl_netzbetreiber,
       p21.total              AS total_2021,
       p24.total              AS total_2024,
       p27.total              AS total_2027,
       m.einwohner, m.dichte, m.flaeche_total, m.beschaeftigte, m.anteil_65plus
FROM gemeinde g
JOIN v_preis_h4 p21 ON p21.gemeinde_nr = g.gemeinde_nr AND p21.jahr = 2021
JOIN v_preis_h4 p24 ON p24.gemeinde_nr = g.gemeinde_nr AND p24.jahr = 2024
JOIN v_preis_h4 p27 ON p27.gemeinde_nr = g.gemeinde_nr AND p27.jahr = 2027
JOIN merkmale  m   ON m.gemeinde_nr   = g.gemeinde_nr;
''')
analyse = pd.read_sql("SELECT * FROM v_analyse", con)
print(f"v_analyse: {len(analyse):,} municipalities")
# Placeholder filter in v_preis_h4: number of 2021 H4 rows left out by the view
ausgefiltert = con.execute('''
    SELECT (SELECT COUNT(*) FROM tarif WHERE jahr = 2021 AND produkt = 'standard' AND kategorie = 'H4')
         - (SELECT SUM(anzahl_netzbetreiber) FROM v_preis_h4 WHERE jahr = 2021)
''').fetchone()[0]
print(f"Placeholder rows left out by v_preis_h4 (2021): {ausgefiltert}")
analyse.head()

v_analyse: 2,087 municipalities
Placeholder rows left out by v_preis_h4 (2021): 10


,gemeinde_nr,gemeinde,anzahl_netzbetreiber,total_2021,total_2024,total_2027,einwohner,dichte,flaeche_total,beschaeftigte,anteil_65plus
0,1,Aeugst am Albis,1,17.122,31.609,23.872,1981.0,250.442478,7.91,442.0,18.021201
1,2,Affoltern am Albis,1,17.122,31.609,23.872,12303.0,1161.756374,10.59,6920.0,18.166301
2,3,Bonstetten,1,17.122,31.609,23.872,5572.0,749.932705,7.43,1014.0,16.529074
3,4,Hausen am Albis,1,17.122,31.609,23.872,3751.0,275.808824,13.60,1021.0,17.995201
4,5,Hedingen,1,17.122,31.609,23.872,3778.0,578.560490,6.53,1478.0,16.728428


## 6 · SQL queries from Python

The queries run with `pd.read_sql(sql, con)` and come back directly as a DataFrame. They show the building blocks the project needs: filtering, grouping, joins and case distinctions.

**6.1 · Price development H4:** mean, minimum and maximum per year. SQLite has no median function, so the median is added in pandas.

In [9]:
entwicklung = pd.read_sql('''
    SELECT jahr,
           COUNT(*)            AS gemeinden,
           ROUND(AVG(total),2) AS mittelwert,
           ROUND(MIN(total),2) AS minimum,
           ROUND(MAX(total),2) AS maximum
    FROM v_preis_h4
    GROUP BY jahr
    ORDER BY jahr
''', con)
median = pd.read_sql("SELECT jahr, total FROM v_preis_h4", con).groupby("jahr")["total"].median().round(2)
entwicklung["median"] = entwicklung["jahr"].map(median)
entwicklung

,jahr,gemeinden,mittelwert,minimum,maximum,median
0,2021,2201,20.90,8.59,27.65,20.63
1,2024,2140,32.00,10.22,50.62,32.15
2,2027,2109,27.24,10.56,45.88,26.91


**6.2 · Which price component moved?** Mean of the components per year in Rp./kWh. This is the basis for sub-question 1.

In [10]:
pd.read_sql('''
    SELECT jahr,
           ROUND(AVG(energie),2)               AS energie,
           ROUND(AVG(netznutzung_inkl_mess),2) AS netznutzung_inkl_mess,
           ROUND(AVG(abgaben),2)               AS abgaben,
           ROUND(AVG(netzzuschlag),2)          AS netzzuschlag,
           ROUND(AVG(total),2)                 AS total
    FROM v_preis_h4
    GROUP BY jahr
    ORDER BY jahr
''', con)

,jahr,energie,netznutzung_inkl_mess,abgaben,netzzuschlag,total
0,2021,7.96,9.79,0.85,2.3,20.90
1,2024,15.59,13.22,0.88,2.3,32.00
2,2027,11.19,12.86,0.90,2.3,27.24


**6.3 · The ten most expensive municipalities in 2027** with a join on `gemeinde` and `netzbetreiber`.

In [11]:
pd.read_sql('''
    SELECT g.name AS gemeinde, n.name AS netzbetreiber, t.total
    FROM tarif t
    JOIN gemeinde      g ON g.gemeinde_nr      = t.gemeinde_nr
    JOIN netzbetreiber n ON n.netzbetreiber_id = t.netzbetreiber_id
    WHERE t.jahr = 2027 AND t.produkt = 'standard' AND t.kategorie = 'H4'
    ORDER BY t.total DESC
    LIMIT 10
''', con)

,gemeinde,netzbetreiber,total
0,Kestenholz,Energie Kestenholz,45.883
1,Niederhelfenschwil,"Politische Gemeinde Niederhelfenschwil, Elektr...",45.817
2,Eggersriet,Elektra- und Wasserkorporation Grub SG,41.683
3,Hauptwil-Gottshaus,Technische Gemeindebetriebe Bischofszell,41.048
4,Niederbüren,Technische Gemeindebetriebe Bischofszell,41.048
5,Hohentannen,Technische Gemeindebetriebe Bischofszell,41.048
6,Bischofszell,Technische Gemeindebetriebe Bischofszell,41.048
7,Zihlschlacht-Sitterdorf,Technische Gemeindebetriebe Bischofszell,41.048
8,Sils im Engadin/Segl,Azienda Elettrica Grevasalvas,40.930
9,Braunau,"Politische Gemeinde Braunau, Elektrizitätsvers...",38.883


**6.4 · Price by municipality size:** municipalities are divided into size classes with `CASE`. This is a first answer to sub-question 2.

In [12]:
pd.read_sql('''
    SELECT CASE
             WHEN einwohner <   1000 THEN '1: under 1000'
             WHEN einwohner <   5000 THEN '2: 1000-4999'
             WHEN einwohner <  20000 THEN '3: 5000-19999'
             ELSE                         '4: 20000 and more'
           END                                    AS groessenklasse,
           COUNT(*)                               AS gemeinden,
           ROUND(AVG(total_2021),2)               AS preis_2021,
           ROUND(AVG(total_2024),2)               AS preis_2024,
           ROUND(AVG(total_2027),2)               AS preis_2027,
           ROUND(AVG(total_2024 - total_2021),2)  AS anstieg_21_24
    FROM v_analyse
    GROUP BY groessenklasse
    ORDER BY groessenklasse
''', con)

,groessenklasse,gemeinden,preis_2021,preis_2024,preis_2027,anstieg_21_24
0,1: under 1000,683,21.49,31.55,27.12,10.06
1,2: 1000-4999,1011,20.74,32.04,27.40,11.30
2,3: 5000-19999,343,20.28,32.61,27.14,12.33
3,4: 20000 and more,50,20.44,32.57,26.84,12.13


**6.5 · Size of the grid operator:** how many municipalities does an operator supply, and is that related to the price? Here a subquery is combined with a join. This is also a feature for sub-question 2.

In [13]:
pd.read_sql('''
    WITH groesse AS (
        SELECT netzbetreiber_id, COUNT(DISTINCT gemeinde_nr) AS anzahl_gemeinden
        FROM tarif
        WHERE jahr = 2027 AND produkt = 'standard' AND kategorie = 'H4'
        GROUP BY netzbetreiber_id
    )
    SELECT CASE
             WHEN gr.anzahl_gemeinden = 1   THEN '1: one municipality'
             WHEN gr.anzahl_gemeinden <= 10 THEN '2: 2-10 municipalities'
             ELSE                                '3: more than 10'
           END                       AS betreibergroesse,
           COUNT(DISTINCT t.netzbetreiber_id) AS betreiber,
           COUNT(*)                  AS tarife,
           ROUND(AVG(t.total),2)     AS preis_2027
    FROM tarif t
    JOIN groesse gr ON gr.netzbetreiber_id = t.netzbetreiber_id
    WHERE t.jahr = 2027 AND t.produkt = 'standard' AND t.kategorie = 'H4'
    GROUP BY betreibergroesse
    ORDER BY betreibergroesse
''', con)

,betreibergroesse,betreiber,tarife,preis_2027
0,1: one municipality,421,421,28.57
1,2: 2-10 municipalities,90,313,27.65
2,3: more than 10,25,1508,26.89


## 7 · Check against Lenny's join

The view `v_analyse` must return the same municipalities and the same prices as Lenny's file `strompreise_gemeinden.csv`. If it does, database and join work with the same numbers.

In [14]:
lenny = pd.read_csv(DATEN / "verbunden" / "strompreise_gemeinden.csv")
vergleich = analyse.merge(lenny, on="gemeinde_nr", how="outer", suffixes=("_db", "_lenny"), indicator=True)

print("Municipalities only in the DB: ", (vergleich["_merge"] == "left_only").sum())
print("Municipalities only at Lenny: ", (vergleich["_merge"] == "right_only").sum())
beide = vergleich[vergleich["_merge"] == "both"]
for spalte in ["total_2021", "total_2024", "total_2027", "einwohner", "dichte",
               "flaeche_total", "beschaeftigte", "anteil_65plus"]:
    a, b = beide[f"{spalte}_db"], beide[f"{spalte}_lenny"]
    abweichung = ((a - b).abs() > 1e-6) | (a.isna() != b.isna())
    print(f"{spalte:<15} deviating values: {abweichung.sum()}")

Municipalities only in the DB:  0
Municipalities only at Lenny:  0
total_2021      deviating values: 0
total_2024      deviating values: 0
total_2027      deviating values: 0
einwohner       deviating values: 0
dichte          deviating values: 0
flaeche_total   deviating values: 0
beschaeftigte   deviating values: 0
anteil_65plus   deviating values: 0


## 8 · How the group continues working with the database

The file `strompreise.db` is all that Subraj and Lenny need. Running this notebook recreates it.

```python
import sqlite3, pandas as pd
con = sqlite3.connect("strompreise.db")
df  = pd.read_sql("SELECT * FROM v_analyse", con)   # finished analysis table
```

- **Subraj (EDA, model):** starts with `v_analyse`. Further features are added to the view as an additional `MAX(CASE WHEN ...)` line.
- **Lenny (map):** joins `gemeinde_nr` with the municipality boundaries.
- **PostgreSQL (Bonus 2):** the schema from section 1 runs almost unchanged. `REAL` becomes `NUMERIC` or `DOUBLE PRECISION`, and `PRAGMA` is dropped.

In [15]:
con.close()
print(f"Done: {DB.relative_to(WURZEL).as_posix()} ({DB.stat().st_size / 1e6:.1f} MB)")

Done: sql/strompreise.db (24.1 MB)
